# SO-101 データセット探索

LeRobot の公開データセットを確認・可視化するノートブックです。

**前提**
- `conda activate so101`
- `jupyter lab notebooks/` で起動

In [ ]:
import os
from pathlib import Path

# HuggingFace Hub から公開 SO-101 データセットを検索
from huggingface_hub import HfApi

api = HfApi()
datasets = api.list_datasets(
    search="so101",
    author="lerobot",
    limit=20,
)

print("=== 公開 SO-101 データセット ===")
for ds in datasets:
    print(f"  {ds.id}")

In [ ]:
# 使用するデータセットを選択
# 例: lerobot/so101_pick_place_lego
REPO_ID = "lerobot/so101_pick_place_lego"  # 適宜変更

In [ ]:
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

# データセットのロード
dataset = LeRobotDataset(REPO_ID)

print(f"データセット: {REPO_ID}")
print(f"エピソード数: {dataset.num_episodes}")
print(f"総フレーム数: {len(dataset)}")
print(f"FPS        : {dataset.fps}")
print(f"特徴量     : {list(dataset.features.keys())}")

In [ ]:
import pandas as pd

# エピソード統計
episode_data = dataset.episode_data_index
df = pd.DataFrame({
    "episode": range(dataset.num_episodes),
    "from": episode_data["from"].numpy(),
    "to": episode_data["to"].numpy(),
})
df["length"] = df["to"] - df["from"]
df["duration_sec"] = df["length"] / dataset.fps

print(f"\nエピソード長 (フレーム数):")
print(df["length"].describe().round(1))
print(f"\n平均エピソード長: {df['duration_sec'].mean():.1f} 秒")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import numpy as np

# エピソード 0 のフレームを可視化
EPISODE_IDX = 0
SAMPLE_FRAMES = [0, 10, 20, 30, 40, 50]  # サンプリングするフレームオフセット

start = dataset.episode_data_index["from"][EPISODE_IDX].item()
end = dataset.episode_data_index["to"][EPISODE_IDX].item()

fig, axes = plt.subplots(1, len(SAMPLE_FRAMES), figsize=(18, 4))
fig.suptitle(f"エピソード {EPISODE_IDX}: フレームサンプル", fontsize=14)

for ax, offset in zip(axes, SAMPLE_FRAMES):
    idx = min(start + offset, end - 1)
    sample = dataset[idx]
    
    # カメラキー名を自動検出
    cam_keys = [k for k in sample.keys() if "image" in k or "camera" in k]
    if cam_keys:
        img = sample[cam_keys[0]]
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).numpy()
            img = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
        ax.imshow(img)
    ax.set_title(f"t={offset}")
    ax.axis("off")

plt.tight_layout()
plt.savefig("notebooks/episode_frames.png", dpi=100, bbox_inches="tight")
plt.show()
print("保存: notebooks/episode_frames.png")

In [ ]:
# アクション (関節角度) の可視化
start = dataset.episode_data_index["from"][EPISODE_IDX].item()
end = dataset.episode_data_index["to"][EPISODE_IDX].item()

actions = []
for idx in range(start, end):
    actions.append(dataset[idx]["action"].numpy())

actions = np.array(actions)
num_joints = actions.shape[1]

fig, axes = plt.subplots(num_joints, 1, figsize=(12, num_joints * 2), sharex=True)
if num_joints == 1:
    axes = [axes]

for j, ax in enumerate(axes):
    ax.plot(actions[:, j])
    ax.set_ylabel(f"Joint {j+1}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("フレーム")
fig.suptitle(f"エピソード {EPISODE_IDX}: 関節角度軌跡", fontsize=14)
plt.tight_layout()
plt.savefig("notebooks/episode_actions.png", dpi=100, bbox_inches="tight")
plt.show()
print("保存: notebooks/episode_actions.png")

In [ ]:
# タスク説明文の確認
if hasattr(dataset, 'meta') and hasattr(dataset.meta, 'tasks'):
    print("タスク説明:")
    for task_id, desc in dataset.meta.tasks.items():
        print(f"  [{task_id}] {desc}")
else:
    sample = dataset[0]
    if "task" in sample:
        print(f"タスク: {sample['task']}")
    elif "language_instruction" in sample:
        print(f"タスク: {sample['language_instruction']}")

In [ ]:
# SmolVLA 学習前の確認チェックリスト
print("=== SmolVLA 学習前チェックリスト ===")
print(f"  [{'OK' if dataset.num_episodes >= 50 else 'NG: 50以上推奨'}] エピソード数: {dataset.num_episodes}")
print(f"  [OK] データセット ID  : {REPO_ID}")
print(f"  [OK] フレーム数       : {len(dataset)}")
print(f"  [OK] FPS              : {dataset.fps}")
print()
print("GCPで学習を開始:")
print("  1. .env の DATASET_NAME を設定")
print("  2. terraform apply (gcp/main.tf)")
print("  3. scp .env user@VM:/workspace/")
print("  4. bash gcp/train_smolvla.sh")